# nb11 — L2-SP Anchored Fine-Tuning

Task 2.2: fine-tune with an L2-SP (L2 Starting Point) regulariser that anchors all trainable
parameters to theta0 (the original poisoned weights), sweeping LAMBDA ∈ {1e-4, 1e-3, 1e-2, 1e-1}.

**Total loss:** `detection_loss + LAMBDA * sum((p - theta0[p])^2)`

No backbone freeze — full model is trainable.  The anchor replaces the freeze as the
stability mechanism, which is more flexible: the optimizer can move in any direction but
is penalised for drifting far from the original weights.

**Expected:** medium LAMBDA (~1e-2) finds the sweet spot — suppresses poison without
destroying real-streak detection.

**Reference from nb10:** LB 284.49 (freeze, 20 iter, no L2SP) and LB 259.79 (baseline, no freeze).

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import copy
import json
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUT_DIR          = Path("/kaggle/working")

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

# Solver
BASE_LR       = 1e-4
MAX_ITER      = 100   # 5x baseline — gives L2-SP time to find the right balance
BATCH_SIZE    = 4
SEED          = 42

# L2-SP sweep
LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1]

# Proxy calibration (from Phase 1, approximated from nb10 run)
SUPPRESSION_REF  = 0.603
PRESERVATION_REF = 0.697  # corrected: nb10 computed 0.697 for the poisoned reference model
PRESERVE_WEIGHT  = 10.0

N_PROBES     = 80
SKY_MEAN_U16 = 8000
STREAK_PEAK  = 45000
PROBES_DIR   = OUT_DIR / "probes"

print(f"Competition data: {COMP_ROOT}")
print(f"L2-SP sweep: LAMBDA in {LAMBDAS}, MAX_ITER={MAX_ITER}")

## Core pipeline + proxy harness

In [ ]:
def build_cfg_base(weights_path, output_dir=None, score_thresh=0.2, seed=42):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                        = str(weights_path)
    cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
    cfg.SEED = seed
    if output_dir:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        dataset_dict["image"]     = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


def compute_iou(box_a, box_b):
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0.0, xi2-xi1) * max(0.0, yi2-yi1)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union = area_a + area_b - inter
    return inter/union if union > 0 else 0.0


def supp_score(predictor, unlearn_dir, iou_thresh=0.2):
    with open(Path(unlearn_dir)/"annotations_coco.json") as f:
        coco = json.load(f)
    img_to_ann = {a["image_id"]: a for a in coco["annotations"]}
    vals = []
    for im_info in coco["images"]:
        ann = img_to_ann.get(im_info["id"])
        if ann is None: vals.append(0.0); continue
        bx,by,bw,bh = ann["bbox"]
        pbox = [bx,by,bx+bw,by+bh]
        im  = load_for_inference(Path(unlearn_dir)/im_info["file_name"])
        out = predictor(im)["instances"].to("cpu")
        best = max(
            (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
             if compute_iou([x1,y1,x2,y2], pbox) >= iou_thresh),
            default=0.0
        )
        vals.append(best)
    return float(np.mean(vals))


def pres_score(predictor, probes_dir):
    with open(Path(probes_dir)/"probes_coco.json") as f:
        coco = json.load(f)
    img_to_anns = {}
    for a in coco["annotations"]:
        img_to_anns.setdefault(a["image_id"], []).append(a)
    vals = []
    for im_info in coco["images"]:
        for ann in img_to_anns.get(im_info["id"], []):
            bx,by,bw,bh = ann["bbox"]
            sbox = [bx,by,bx+bw,by+bh]
            im  = load_for_inference(Path(probes_dir)/im_info["file_name"])
            out = predictor(im)["instances"].to("cpu")
            best = max(
                (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
                 if compute_iou([x1,y1,x2,y2], sbox) >= 0.2),
                default=0.0
            )
            vals.append(best)
    return float(np.mean(vals)) if vals else 0.0


def proxy(supp_now, pres_now,
          supp_ref=SUPPRESSION_REF, pres_ref=PRESERVATION_REF, w=PRESERVE_WEIGHT):
    gain = supp_ref - supp_now
    loss = max(0.0, pres_ref - pres_now)
    return gain - w*loss, gain, loss

## Generate probes (SEED=42, MARGIN=10 — matches nb10 calibration)

In [ ]:
PROBES_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

def draw_streak(img_u16, x0, y0, x1, y1, sigma=3.0, peak=45000):
    dx,dy = x1-x0, y1-y0
    length = math.hypot(dx,dy)
    if length < 1: return None
    pad = int(sigma*3)+1
    bx0,by0 = max(0,min(x0,x1)-pad), max(0,min(y0,y1)-pad)
    bx1,by1 = min(IMG_W,max(x0,x1)+pad), min(IMG_H,max(y0,y1)+pad)
    ys,xs = np.mgrid[by0:by1,bx0:bx1]
    t = ((xs-x0)*dx+(ys-y0)*dy)/(length*length)
    tc = np.clip(t,0,1)
    dist = np.sqrt((xs-x0-tc*dx)**2+(ys-y0-tc*dy)**2)
    g = np.exp(-0.5*(dist/sigma)**2)
    taper = np.where(t<0,np.exp(-2*t**2),np.where(t>1,np.exp(-2*(t-1)**2),1.0))
    img_u16[by0:by1,bx0:bx1] = np.clip(
        img_u16[by0:by1,bx0:bx1].astype(np.float32)+g*taper*peak,0,65535).astype(np.uint16)
    return [bx0,by0,bx1-bx0,by1-by0]

coco_imgs,coco_anns,ann_id = [],[],1
for idx in range(N_PROBES):
    bg = rng.normal(SKY_MEAN_U16,300,(IMG_H,IMG_W)).clip(0,65535).astype(np.uint16)
    n_s = 1 if idx < int(N_PROBES*0.7) else 2
    bboxes = []
    for _ in range(n_s):
        bw,bh = int(rng.integers(20,81)), int(rng.integers(20,81))
        angle = rng.uniform(0,math.pi)
        cx = int(rng.integers(10+bw//2,IMG_W-10-bw//2))
        cy = int(rng.integers(10+bh//2,IMG_H-10-bh//2))
        hl = math.hypot(bw,bh)/2
        x0,y0 = int(cx-hl*math.cos(angle)), int(cy-hl*math.sin(angle))
        x1,y1 = int(cx+hl*math.cos(angle)), int(cy+hl*math.sin(angle))
        b = draw_streak(bg,x0,y0,x1,y1)
        if b: bboxes.append(b)
    fname = f"probe_{idx:04d}.png"
    cv2.imwrite(str(PROBES_DIR/fname), bg)
    iid = idx+1
    coco_imgs.append({"id":iid,"file_name":fname,"height":IMG_H,"width":IMG_W})
    for b in bboxes:
        coco_anns.append({"id":ann_id,"image_id":iid,"category_id":1,
                          "bbox":[float(v) for v in b],"area":float(b[2]*b[3]),"iscrowd":0})
        ann_id += 1

with open(PROBES_DIR/"probes_coco.json","w") as f:
    json.dump({"images":coco_imgs,"annotations":coco_anns,
               "categories":[{"id":1,"name":"streak"}]}, f)
print(f"Generated {N_PROBES} probes, {len(coco_anns)} annotations")

## L2-SP Trainer

Overrides `DefaultTrainer.run_step()` to inject the L2-SP penalty into the total loss.
theta0 is saved AFTER `resume_or_load()` to capture the actual poisoned weights.

In [ ]:
_L2SP_LAMBDA = 0.0   # set before each training run
_L2SP_THETA0 = {}    # {name: cpu_tensor} — set after resume_or_load


class L2SPTrainer(DefaultTrainer):
    """DefaultTrainer + empty-annotation data loader + L2-SP loss."""

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    def run_step(self):
        assert self.model.training
        data = next(self._trainer._data_loader_iter)

        loss_dict = self.model(data)

        # L2-SP penalty: sum of squared distances from original poisoned weights
        device = next(self.model.parameters()).device
        l2sp = sum(
            (param - _L2SP_THETA0[name].to(device)).pow(2).sum()
            for name, param in self.model.named_parameters()
            if name in _L2SP_THETA0
        )

        losses = sum(loss_dict.values()) + _L2SP_LAMBDA * l2sp

        self.optimizer.zero_grad()
        losses.backward()
        self.optimizer.step()


UNLEARN_DATASET = "unlearn"

def _register_unlearn():
    if UNLEARN_DATASET in DatasetCatalog.list():
        return
    with open(Path(UNLEARN_DIR)/"annotations_coco.json") as f:
        coco = json.load(f)
    dicts = [
        {"file_name": str(Path(UNLEARN_DIR)/im["file_name"]),
         "height":    im["height"], "width": im["width"],
         "image_id":  im["id"],     "annotations": []}
        for im in coco["images"]
    ]
    DatasetCatalog.register(UNLEARN_DATASET, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered '{UNLEARN_DATASET}': {len(dicts)} images")

_register_unlearn()

## Reference proxy (poisoned model, no training)

In [ ]:
cfg_ref = build_cfg_base(POISONED_WEIGHTS, score_thresh=CONF_THRESH, seed=SEED)
pred_ref = DefaultPredictor(cfg_ref)

supp_ref_val  = supp_score(pred_ref, UNLEARN_DIR)
pres_ref_val  = pres_score(pred_ref, str(PROBES_DIR))
prx_ref, g_r, l_r = proxy(supp_ref_val, pres_ref_val)

print(f"Reference (poisoned model):")
print(f"  suppression  = {supp_ref_val:.4f}")
print(f"  preservation = {pres_ref_val:.4f}")
print(f"  proxy        = {prx_ref:.4f}  (gain={g_r:.4f}, loss={l_r:.4f})")

## L2-SP lambda sweep

In [ ]:
import gc
global _L2SP_LAMBDA, _L2SP_THETA0

results = {
    "reference": {
        "lambda": None, "suppression": supp_ref_val, "preservation": pres_ref_val,
        "proxy": prx_ref, "supp_gain": g_r, "pres_loss": l_r,
    }
}

for lam in LAMBDAS:
    key = f"lambda_{lam:.0e}"
    print(f"\n{'='*60}")
    print(f"Training LAMBDA={lam:.0e}, MAX_ITER={MAX_ITER}")
    print(f"{'='*60}")

    out_dir = OUT_DIR / f"model_l2sp_{lam:.0e}"
    cfg_m = build_cfg_base(POISONED_WEIGHTS, out_dir, score_thresh=CONF_THRESH, seed=SEED)
    cfg_m.DATASETS.TRAIN   = (UNLEARN_DATASET,)
    cfg_m.DATASETS.TEST    = ()
    cfg_m.DATALOADER.NUM_WORKERS = 2
    cfg_m.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
    cfg_m.SOLVER.BASE_LR         = BASE_LR
    cfg_m.SOLVER.MAX_ITER        = MAX_ITER
    cfg_m.SOLVER.STEPS           = []

    _L2SP_LAMBDA = lam
    trainer = L2SPTrainer(cfg_m)
    trainer.resume_or_load(resume=False)

    # Capture theta0 AFTER loading the actual poisoned weights
    _L2SP_THETA0 = {
        name: param.data.clone().cpu()
        for name, param in trainer.model.named_parameters()
    }
    print(f"  theta0 captured: {len(_L2SP_THETA0)} parameter tensors")

    trainer.train()

    # Load trained model
    cfg_infer = build_cfg_base(
        out_dir/"model_final.pth", score_thresh=CONF_THRESH, seed=SEED
    )
    pred_m = DefaultPredictor(cfg_infer)

    supp_m  = supp_score(pred_m, UNLEARN_DIR)
    pres_m  = pres_score(pred_m, str(PROBES_DIR))
    prx_m, g_m, l_m = proxy(supp_m, pres_m)

    print(f"  suppression  = {supp_m:.4f}  (gain={g_m:.4f})")
    print(f"  preservation = {pres_m:.4f}  (loss={l_m:.4f})")
    print(f"  PROXY        = {prx_m:.4f}")

    results[key] = {
        "lambda": lam, "suppression": supp_m, "preservation": pres_m,
        "proxy": prx_m, "supp_gain": g_m, "pres_loss": l_m,
    }

    del trainer, pred_m
    gc.collect()
    torch.cuda.empty_cache()

_L2SP_THETA0 = {}  # clear memory
print("\nSweep complete.")

## Results table

In [ ]:
print("\n=== PHASE 2 TASK 2.2 — L2-SP LAMBDA SWEEP ===")
print(f"{'Config':<22} {'Lambda':>8} {'Supp':>7} {'S_gain':>8} {'Pres':>7} {'P_loss':>8} {'PROXY':>8}")
print("-" * 75)
for key, r in results.items():
    lam_str = f"{r['lambda']:.0e}" if r['lambda'] is not None else "—"
    print(f"{key:<22} {lam_str:>8} {r['suppression']:>7.4f} {r['supp_gain']:>8.4f} "
          f"{r['preservation']:>7.4f} {r['pres_loss']:>8.4f} {r['proxy']:>8.4f}")

# Best proxy among trained models
trained = {k:v for k,v in results.items() if k != 'reference'}
best_key = max(trained, key=lambda k: trained[k]['proxy'])
best = results[best_key]
print(f"\nBest: {best_key}  proxy={best['proxy']:.4f}  lambda={best['lambda']:.0e}")

with open(OUT_DIR/"nb11_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved nb11_results.json")

## Inference with best lambda → submission.csv

In [ ]:
best_lam = best['lambda']
best_weights = OUT_DIR / f"model_l2sp_{best_lam:.0e}" / "model_final.pth"

cfg_best = build_cfg_base(best_weights, score_thresh=CONF_THRESH, seed=SEED)
pred_best = DefaultPredictor(cfg_best)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Inference on {len(test_files)} test images (best lambda={best_lam:.0e})...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_for_inference(img_path)
    out = pred_best(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts = []
    for (x1,y1,x2,y2),s in zip(boxes, scores):
        x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
        x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
        w,h = max(0.0,x2-x1), max(0.0,y2-y1)
        if w==0 or h==0: continue
        parts.extend([f"{float(s):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"])
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(OUT_DIR/"submission.csv", index=False)
print(f"Wrote submission.csv  ({len(submission)} rows)")

assert len(submission) == 2000
empty = (submission["prediction_string"].str.strip().fillna("")=="").sum()
total_dets = submission["prediction_string"].apply(
    lambda s: len(str(s).split())//5 if isinstance(s,str) and str(s).strip() else 0
).sum()
print(f"  Empty rows: {empty}  Total dets: {total_dets}")
print("Best lambda:", best_lam, "  Proxy:", round(best['proxy'],4))
print("Ready to submit.")